# Import Modules

In [ ]:
import math
import requests
import folium
import requests
import numpy as np
import gravis as gv
import pandas as pd
import networkx as nx

from tqdm import tqdm
from scipy.spatial import KDTree

# Load data

In [17]:
# Load data
stops_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stops.txt")
stop_times_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stop_times.txt")
trips = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\trips.txt")

# Convert datatypes
stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

# Merge route id details into stop times
stop_times_df = stop_times_df.merge(
    trips[["route_id", "trip_id"]],
    left_on='trip_id', 
    right_on='trip_id', 
    how='left')

# # Isolate Bradford
# stops_df = stops_df.loc[(stops_df["stop_lat"] < 54) & (stops_df["stop_lat"] > 53.7)].reset_index(drop=True)
# stops_df = stops_df.loc[(stops_df["stop_lon"] < -1.55) & (stops_df["stop_lon"] > -1.95)].reset_index(drop=True)

In [18]:
stops_loc_dict = stops_df.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")

# Get longest trips in each direction for each unique route 

In [19]:
grouped = stop_times_df.groupby("trip_id")["stop_sequence"]
stop_times_df["trip_len"] = (grouped.transform("max") - grouped.transform("min"))

In [20]:
trip_stops_dict = stop_times_df.groupby('trip_id')['stop_id'].apply(set).to_dict()

# 2. Extract trip-level information to reduce dimensionality
# We drop from millions of stop-time rows down to just the unique trips
trip_info = stop_times_df[['route_id', 'trip_id', 'stop_sequence', 'trip_len']].drop_duplicates()

# Sort descending once so that when we group, the longest trip is always first
trip_info = trip_info.sort_values(by=['route_id', 'trip_len'], ascending=[True, False])

In [21]:
include_ids = []
considered_routes = set()

# 3. Iterate over grouped trips rather than the full stop_times DataFrame
for route_id, route_trips in tqdm(trip_info.groupby('route_id')):
    if route_id in considered_routes:
        print(f"Already consided route {route_id}, idk how it's ended up here again...")
    considered_routes.add(route_id)
    # Skip if route has no trips (edge case)
    if route_trips.empty:
        continue
        
    # Because we sorted earlier, the longest trip is guaranteed to be the first row
    longest_trip_id = route_trips.iloc[0]['trip_id']
    include_ids.append(longest_trip_id)
    
    # Fetch pre-computed stops for the longest trip
    longest_trip_stops = list(trip_stops_dict[longest_trip_id])
    
    # Get two random stops (handle case where trip has < 2 stops)
    sample_stops = set(np.random.choice(longest_trip_stops, 2, replace=False))
        
    # Find the next longest trip not containing these stops
    # We slice [1:] to skip the longest trip we already evaluated
    for trip_id in route_trips['trip_id'].iloc[1:]:
        cur_stops = trip_stops_dict[trip_id]
        
        # isdisjoint() is highly optimized in C for Python sets
        if cur_stops.isdisjoint(sample_stops):
            include_ids.append(trip_id)
            break

# 4. Final single filter step
longest_trips_df = stop_times_df[stop_times_df["trip_id"].isin(include_ids)].reset_index(drop=True)

100%|██████████| 1140/1140 [00:01<00:00, 866.46it/s]


# Tranfer cost graph

In [22]:
# 1. Use a Directed Graph. This is strictly required so that 
# passengers cannot use 0-cost alighting edges backwards to board.
G = nx.MultiDiGraph() 

TRANSFER_PENALTY_SEC = 600  # 10 minute penalty for boarding a new route

# 2. Ensure data is strictly ordered by trip and sequence
longest_trips_df = longest_trips_df.sort_values(["trip_id", "stop_sequence"])

# 3. Group by TRIP, not route
for trip_id, trip_to_plot in longest_trips_df.groupby("trip_id"):
    trip_to_plot = trip_to_plot.reset_index(drop=True)
    
    # Extract the route_id for this specific trip to label the platforms
    route_id = trip_to_plot["route_id"].iloc[0]

    for i in range(len(trip_to_plot) - 1):
        cur_stop = trip_to_plot.loc[i, "stop_id"]
        next_stop = trip_to_plot.loc[i+1, "stop_id"]

        # Define platform nodes as tuples
        cur_platform = (cur_stop, route_id)
        next_platform = (next_stop, route_id)

        duration = trip_to_plot.loc[i+1, "arrival_time"] - trip_to_plot.loc[i, "departure_time"]
        duration_sec = duration.total_seconds()
        
        # Guard against negative time loops
        if duration_sec < 0:
            continue

        # --- 1. Travel Edge (Platform to Platform) ---
        # Directed edge: Train only goes one way
        G.add_edge(cur_platform, next_platform, weight=duration_sec)
        
        # --- 2. Boarding Edges (Station -> Platform) ---
        # Directed edge: Station to Platform costs the penalty
        if not G.has_edge(cur_stop, cur_platform):
            G.add_edge(cur_stop, cur_platform, weight=TRANSFER_PENALTY_SEC)
            
        if not G.has_edge(next_stop, next_platform):
            G.add_edge(next_stop, next_platform, weight=TRANSFER_PENALTY_SEC)
            
        # --- 3. Alighting Edges (Platform -> Station) ---
        # Directed edge: Platform to Station is free
        if not G.has_edge(cur_platform, cur_stop):
            G.add_edge(cur_platform, cur_stop, weight=0)
            
        if not G.has_edge(next_platform, next_stop):
            G.add_edge(next_platform, next_stop, weight=0)

## Calcualte walks between stops

In [23]:
# docker run -t -i -p 5001:5000 -v "${PWD}:/data" osrm/osrm-backend osrm-routed --algorithm mld /data/west-yorkshire-foot.osrm

In [24]:
# --- PREPARE SPATIAL INDEX ---
# Assuming stops_df has columns: 'stop_id', 'stop_lat', 'stop_lon'
stop_ids = stops_df['stop_id'].tolist()
coordinates = stops_df[['stop_lat', 'stop_lon']].values

# Build a KDTree for lightning-fast geometric nearest-neighbor lookups
spatial_tree = KDTree(coordinates)

def get_walking_durations_batch(source_idx, target_indices, stops_df):
    """
    Queries OSRM Table API to get durations from 1 source to multiple destinations
    in a single HTTP request.
    """
    # Grab rows from dataframe
    source_row = stops_df.iloc[source_idx]
    target_rows = stops_df.iloc[target_indices]
    
    # OSRM expects: lon,lat
    source_coord = f"{source_row['stop_lon']},{source_row['stop_lat']}"
    target_coords = [f"{row['stop_lon']},{row['stop_lat']}" for _, row in target_rows.iterrows()]
    
    # Combine into a single coordinate string string: source;dest1;dest2...
    all_coords = ";".join([source_coord] + target_coords)
    
    # We set sources=0 (the first coordinate) so it only calculates rows radiating OUT from the source
    url = f"http://127.0.0.1:5001/table/v1/foot/{all_coords}?sources=0"
    
    try:
        response = requests.get(url).json()
        # response['durations'][0] will be an array: [dist_to_self, dist_to_dest1, dist_to_dest2, ...]
        # We skip index 0 because it's the distance to itself
        return response['durations'][0][1:]
    except Exception as e:
        # Handle failed requests safely
        return [None] * len(target_indices)

In [25]:
def time_journey(origin, dest):
    o_str = f"{origin[1]},{origin[0]}"
    d_str = f"{dest[1]},{dest[0]}"
    
    url = f"http://127.0.0.1:5001/route/v1/foot/{o_str};{d_str}?overview=false"
    
    response = requests.get(url).json()
    return response['routes'][0]['duration']

def time_walk_between_stops(origin_stop_id, dest_stop_id):
    origin = list(stops_loc_dict[origin_stop_id].values())
    dest = list(stops_loc_dict[dest_stop_id].values())

    return time_journey(origin, dest)

time_walk_between_stops("450014118", "450021125")

7512.6

In [26]:
WALK_NEIGHBORS = 11

print("Calculating and adding walking transfers...")
for idx, row in tqdm(stops_df.iterrows(), total=len(stops_df)):
    current_stop_id = row['stop_id']
    current_coords = [row['stop_lat'], row['stop_lon']]
    
    # 1. Find the nearest 11 candidates geometrically (inc. self)
    distances, indices = spatial_tree.query(current_coords, k=WALK_NEIGHBORS)
    
    # Remove the first item (which is the current node filtering itself out)
    target_indices = indices[1:]
    
    # 2. Get real walking durations along actual footpaths from OSRM
    durations = get_walking_durations_batch(idx, target_indices, stops_df)
    
    # 3. Add walking edges to your Directed Graph
    for target_idx, duration_sec in zip(target_indices, durations):
        if duration_sec is None:
            continue
            
        neighbor_stop_id = stops_df.iloc[target_idx]['stop_id']
        
        # Add walking edge in BOTH directions (since walking paths are bidirectional)
        # Note: We add them directly between the base string IDs
        G.add_edge(current_stop_id, neighbor_stop_id, weight=duration_sec, type="walk")
        G.add_edge(neighbor_stop_id, current_stop_id, weight=duration_sec, type="walk")

11
Calculating and adding walking transfers...


100%|██████████| 31997/31997 [01:51<00:00, 286.12it/s]


### Set node position attibutes

In [27]:
node_attributes = {}

for node in G.nodes:
    if isinstance(node, tuple):
        # This is a Platform node: (stop_id, route_id)
        parent_stop_id = node[0]
        if parent_stop_id in stops_loc_dict:
            node_attributes[node] = {
                "lat": stops_loc_dict[parent_stop_id]["stop_lat"], 
                "lon": stops_loc_dict[parent_stop_id]["stop_lon"]
            }
    else:
        # This is a Station node: stop_id
        if node in stops_loc_dict:
            node_attributes[node] = {
                "lat": stops_loc_dict[node]["stop_lat"], 
                "lon": stops_loc_dict[node]["stop_lon"]
            }

nx.set_node_attributes(G, node_attributes)

# Search Graph

In [28]:
MEAN_LAT_RAD = math.radians(53.8) 
LON_SCALE = math.cos(MEAN_LAT_RAD)

METERS_PER_DEGREE = 111320.0
MAX_SPEED_MPS = 25.0

def manhattan_heuristic(node1, node2):
    stop1 = node1[0] if isinstance(node1, tuple) else node1
    stop2 = node2[0] if isinstance(node2, tuple) else node2

    lat1, lon1 = G.nodes[stop1]['lat'], G.nodes[stop1]['lon']
    lat2, lon2 = G.nodes[stop2]['lat'], G.nodes[stop2]['lon']

    
    dlat = abs(lat2 - lat1)
    dlon = abs(lon2 - lon1) * LON_SCALE
    
    # distance = METERS_PER_DEGREE * (dlat + dlon)
    
    return (dlat + dlon)# distance / MAX_SPEED_MPS

In [29]:
start_stop_id = "450014118"
end_stop_id = "450021125"

try:
    optimal_path = nx.astar_path(
        G=G,
        source=start_stop_id,
        target=end_stop_id,
        heuristic=manhattan_heuristic,
        weight="weight"
    )
    
    optimal_duration = nx.astar_path_length(
        G=G,
        source=start_stop_id,
        target=end_stop_id,
        heuristic=manhattan_heuristic,
        weight="weight"
    )
    
    print(f"Path: {optimal_path}")
    print(f"Total Duration: {optimal_duration//60} mins")

except nx.NetworkXNoPath:
    print("No path exists between the specified stops.")
except nx.NodeNotFound as e:
    print(f"Node missing from graph: {e}")

Path: ['450014118', ('450014118', np.int64(134302)), ('450028737', np.int64(134302)), ('450015846', np.int64(134302)), ('450018825', np.int64(134302)), ('450018820', np.int64(134302)), ('450018814', np.int64(134302)), ('450018838', np.int64(134302)), ('450018839', np.int64(134302)), ('450024300', np.int64(134302)), ('450024304', np.int64(134302)), ('450018843', np.int64(134302)), ('450018845', np.int64(134302)), ('450018849', np.int64(134302)), ('450018850', np.int64(134302)), ('450018853', np.int64(134302)), ('450018855', np.int64(134302)), ('450018856', np.int64(134302)), ('450018859', np.int64(134302)), ('450027447', np.int64(134302)), ('450018860', np.int64(134302)), ('450024361', np.int64(134302)), '450024361', '450024365', ('450024365', np.int64(22577)), ('450018644', np.int64(22577)), ('450018642', np.int64(22577)), ('450018649', np.int64(22577)), ('450018650', np.int64(22577)), ('450020527', np.int64(22577)), ('450020531', np.int64(22577)), ('450020532', np.int64(22577)), ('450

# Draw resulting route

In [30]:
AVAILABLE_COLORS = [
    'blue', 'red', 'green', 'purple', 'orange', 
    'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'pink'
]

segments = []
current_type = None
current_route = None
current_stops = []
current_duration = 0

# 1. Segment the path, retaining stop IDs, handling walk transfers, and calculating durations
for i in range(len(optimal_path) - 1):
    u = optimal_path[i]
    v = optimal_path[i+1]
    
    # Retrieve edge data to calculate durations. Handle MultiDiGraph by taking min weight edge.
    edges = G.get_edge_data(u, v)
    edge_data = min(edges.values(), key=lambda x: x.get('weight', float('inf')))
    weight = edge_data.get('weight', 0)
    
    u_is_plat = isinstance(u, tuple)
    v_is_plat = isinstance(v, tuple)
    
    # Categorize the edge
    if u_is_plat and v_is_plat:
        seg_type = "transit"
        route = u[1]
    elif not u_is_plat and not v_is_plat:
        seg_type = "walk"
        route = "walk"
    else:
        # Transfer edge (Boarding or Alighting at the same coordinate)
        if not u_is_plat and v_is_plat:
            # Boarding (Station -> Platform): Apply boarding penalty time to the upcoming transit segment
            seg_type = "transit"
            route = v[1]
        else:
            # Alighting (Platform -> Station): Append alighting duration to the completed transit segment
            if segments:
                segments[-1]["duration"] += weight
            continue 
            
    # If the travel mode or route changes, finalize the current segment and start a new one
    if seg_type != current_type or route != current_route:
        if current_stops:
            segments.append({
                "type": current_type,
                "route_id": current_route,
                "stops": current_stops,
                "duration": current_duration
            })
        current_type = seg_type
        current_route = route
        current_stops = [{"coord": (G.nodes[u]['lat'], G.nodes[u]['lon']), "id": u}]
        current_duration = 0
        
    current_stops.append({"coord": (G.nodes[v]['lat'], G.nodes[v]['lon']), "id": v})
    current_duration += weight

# Append the final segment
if current_stops:
    segments.append({
        "type": current_type,
        "route_id": current_route,
        "stops": current_stops,
        "duration": current_duration
    })

unique_routes = {seg["route_id"] for seg in segments if seg["type"] == "transit"}
route_colors = {
    route: AVAILABLE_COLORS[i % len(AVAILABLE_COLORS)] 
    for i, route in enumerate(unique_routes)
}

# 2. Initialize the map
start_node = optimal_path[0]
start_lat, start_lon = G.nodes[start_node]['lat'], G.nodes[start_node]['lon']
route_map = folium.Map(location=[start_lat, start_lon], zoom_start=14, tiles='CartoDB positron')

# 3. Plot segments and node dots
for seg in segments:
    line_coords = [stop["coord"] for stop in seg["stops"]]
    duration_mins = int(seg["duration"] // 60)
    
    if seg["type"] == "transit":
        color = route_colors[seg["route_id"]]
        tooltip_text = f"Route: {seg['route_id']} | Duration: {duration_mins} mins"
        dash_array = None
        line_weight = 6
    else:
        color = "gray"
        tooltip_text = f"Walk | Duration: {duration_mins} mins"
        dash_array = "5, 5"
        line_weight = 4

    # Plot the segment line
    folium.PolyLine(
        locations=line_coords,
        color=color,
        weight=line_weight,
        opacity=0.8,
        dash_array=dash_array,
        tooltip=tooltip_text
    ).add_to(route_map)
    
    # Plot a dot for each stop on this segment
    for stop in seg["stops"]:
        stop_id_display = stop['id'][0] if isinstance(stop['id'], tuple) else stop['id']
        folium.CircleMarker(
            location=stop["coord"],
            radius=4,
            color=color,
            fill=True,
            fill_color="white",
            fill_opacity=1,
            weight=2,
            tooltip=f"Stop: {stop_id_display}"
        ).add_to(route_map)

# 4. Add prominent markers for Start, End, and Transfers
folium.Marker(
    location=(start_lat, start_lon),
    popup="Start",
    icon=folium.Icon(color="green", icon="play")
).add_to(route_map)

end_node = optimal_path[-1]
folium.Marker(
    location=(G.nodes[end_node]['lat'], G.nodes[end_node]['lon']),
    popup="Destination",
    icon=folium.Icon(color="red", icon="stop")
).add_to(route_map)

# Add distinct black transfer dots over the transfer points
for i in range(1, len(segments)):
    transfer_coord = segments[i]["stops"][0]["coord"]
    folium.CircleMarker(
        location=transfer_coord,
        radius=6,
        color="black",
        fill=True,
        fill_color="black",
        fill_opacity=1,
        tooltip="Transfer Point"
    ).add_to(route_map)

route_map.save("a_star_optimal_route.html")